# Phase 7: GPS 2D Pre-training (300k, Kaggle T4)
Graph Transformer on 2D bond-topology graphs.
Outputs: `gps_2d_encoder.pt` + `gps_2d_best.pt` + embeddings

In [ ]:
!pip install -q torch-geometric optuna

In [ ]:
import torch, os, json, time
import numpy as np
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GPSConv, GINEConv, global_mean_pool
import torch.nn as nn
import optuna

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'{props.name} | {props.total_memory/1e9:.1f} GB')

SAVE_DIR = '/kaggle/working'

# Auto-detect input path
for candidate in [
    '/kaggle/input/pyg-2d-graphs-bond-300k',
    '/kaggle/input/datasets/nothingnessvoid/pyg-2d-graphs-bond-300k',
]:
    if os.path.isdir(candidate):
        INPUT_DIR = candidate
        break
else:
    raise FileNotFoundError('Dataset not found')

print('Input:', INPUT_DIR)
print('Save:', SAVE_DIR)

In [ ]:
# Load 2D graphs
graphs = torch.load(f'{INPUT_DIR}/pyg_2d_graphs_bond_300k.pt', weights_only=False)
print(f'Loaded {len(graphs)} 2D graphs')
print(f'Node features: {graphs[0].x.shape[1]}, Edge features: {graphs[0].edge_attr.shape[1]}')

SEED = 42
N = len(graphs)
idx = np.random.RandomState(SEED).permutation(N)

# Full splits (for retrain + test + embeddings)
n_train = int(0.8 * N)
n_val = int(0.1 * N)
train_set_full = [graphs[i] for i in idx[:n_train]]
val_set_full = [graphs[i] for i in idx[n_train:n_train+n_val]]
test_set = [graphs[i] for i in idx[n_train+n_val:]]

# Small splits for Optuna (50k total -> 40k train, 5k val)
OPTUNA_LIMIT = 50000
idx_small = np.random.RandomState(SEED).permutation(OPTUNA_LIMIT)
n_train_s = int(0.8 * OPTUNA_LIMIT)
n_val_s = int(0.1 * OPTUNA_LIMIT)
train_set_small = [graphs[i] for i in idx_small[:n_train_s]]
val_set_small = [graphs[i] for i in idx_small[n_train_s:n_train_s+n_val_s]]

print(f'Full split: train={len(train_set_full)}, val={len(val_set_full)}, test={len(test_set)}')
print(f'Optuna split: train={len(train_set_small)}, val={len(val_set_small)}')

In [ ]:
class GPSWrapper(nn.Module):
    def __init__(self, in_channels=9, edge_dim=4, hidden_channels=128,
                 num_layers=6, num_heads=8, dropout=0.1, n_targets=3):
        super().__init__()
        self.node_emb = nn.Linear(in_channels, hidden_channels)
        self.edge_emb = nn.Linear(edge_dim, hidden_channels)
        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            gin = GINEConv(
                nn.Sequential(
                    nn.Linear(hidden_channels, hidden_channels),
                    nn.SiLU(),
                    nn.Linear(hidden_channels, hidden_channels),
                ), edge_dim=hidden_channels)
            gps = GPSConv(channels=hidden_channels, conv=gin,
                          heads=num_heads, dropout=dropout,
                          act='silu', norm='batch_norm',
                          attn_type='multihead')
            self.convs.append(gps)
        self.hidden_channels = hidden_channels
        self.head = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.SiLU(),
            nn.Linear(hidden_channels // 2, n_targets),
        )

    def forward(self, x, edge_index, edge_attr, batch):
        return self.head(self.encode(x, edge_index, edge_attr, batch))

    def encode(self, x, edge_index, edge_attr, batch):
        h = self.node_emb(x.float())
        e = self.edge_emb(edge_attr.float())
        for conv in self.convs:
            h = conv(h, edge_index, batch, edge_attr=e)
        return global_mean_pool(h, batch)

print('GPSWrapper defined')

In [ ]:
def run_training(params, train_set, val_set, max_epochs=80,
                 patience=15, resume_ckpt=None, save_prefix='gps'):
    model = GPSWrapper(
        hidden_channels=params['hidden_channels'],
        num_layers=params['num_layers'],
        num_heads=params['num_heads'],
        dropout=params['dropout'],
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=params['lr'],
                                  weight_decay=params['weight_decay'])
    if params['scheduler'] == 'plateau':
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, patience=5, factor=0.5, min_lr=1e-6)
    else:
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max_epochs, eta_min=1e-6)

    scaler = torch.amp.GradScaler()
    criterion = nn.L1Loss()

    start_epoch = 0
    if resume_ckpt and os.path.exists(resume_ckpt):
        ckpt = torch.load(resume_ckpt, weights_only=False)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        sched.load_state_dict(ckpt['scheduler'])
        scaler.load_state_dict(ckpt['scaler'])
        start_epoch = ckpt['epoch'] + 1
        print(f'Resumed from epoch {start_epoch}')

    bs = params['batch_size']
    train_loader = DataLoader(train_set, batch_size=bs, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=bs, shuffle=False,
                            num_workers=2, pin_memory=True)

    best_val = float('inf')
    wait = 0

    for epoch in range(start_epoch, max_epochs):
        model.train()
        t0 = time.time()
        train_loss = 0.0
        for batch_data in train_loader:
            batch_data = batch_data.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                pred = model(batch_data.x, batch_data.edge_index,
                             batch_data.edge_attr, batch_data.batch)
                loss = criterion(pred, batch_data.y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item() * batch_data.num_graphs
        train_loss /= len(train_set)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_data in val_loader:
                batch_data = batch_data.to(device)
                with torch.amp.autocast('cuda'):
                    pred = model(batch_data.x, batch_data.edge_index,
                                 batch_data.edge_attr, batch_data.batch)
                    loss = criterion(pred, batch_data.y)
                val_loss += loss.item() * batch_data.num_graphs
        val_loss /= len(val_set)

        if params['scheduler'] == 'plateau':
            sched.step(val_loss)
        else:
            sched.step()

        elapsed = time.time() - t0
        lr_now = optimizer.param_groups[0]['lr']
        print(f'  ep{epoch:03d} train={train_loss:.4f} val={val_loss:.4f} '
              f'lr={lr_now:.2e} {elapsed:.0f}s')

        if val_loss < best_val:
            best_val = val_loss
            wait = 0
            torch.save(model.state_dict(),
                       f'{SAVE_DIR}/{save_prefix}_best.pt')
        else:
            wait += 1

        if epoch % 5 == 0:
            torch.save({
                'epoch': epoch, 'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': sched.state_dict(),
                'scaler': scaler.state_dict(),
                'best_val': best_val,
            }, f'{SAVE_DIR}/{save_prefix}_ckpt.pt')

        if wait >= patience:
            print(f'  Early stop at epoch {epoch}')
            break

    return best_val

print('run_training defined')

In [ ]:
# Best params from previous Optuna run (20 trials on 50k subset)
# Skip Optuna search — use hardcoded results directly
best_params = {
    'hidden_channels': 192,
    'num_layers': 7,
    'num_heads': 4,
    'dropout': 0.05,
    'lr': 0.0004754654349367296,
    'weight_decay': 1.3094136884618282e-05,
    'batch_size': 256,
    'scheduler': 'cosine',
}
print('Using saved best params (Optuna search skipped):')
print(f'  MAE=0.1445 (on 50k subset)')
print(best_params)

In [ ]:
# Retrain best on FULL 300k data
bp = best_params
print('\nRetraining best config on full 300k...')
best_val = run_training(bp, train_set_full, val_set_full,
                        max_epochs=150, patience=25,
                        save_prefix='gps_2d_best')

In [ ]:
# Test evaluation
bp = best_params
model = GPSWrapper(
    hidden_channels=bp['hidden_channels'],
    num_layers=bp['num_layers'],
    num_heads=bp['num_heads'],
    dropout=bp['dropout'],
).to(device)
model.load_state_dict(torch.load(f'{SAVE_DIR}/gps_2d_best_best.pt',
                                  weights_only=False))
model.eval()

test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)
from sklearn.metrics import mean_absolute_error, r2_score

all_pred, all_true = [], []
with torch.no_grad():
    for batch_data in test_loader:
        batch_data = batch_data.to(device)
        pred = model(batch_data.x, batch_data.edge_index,
                     batch_data.edge_attr, batch_data.batch)
        all_pred.append(pred.cpu().numpy())
        all_true.append(batch_data.y.cpu().numpy())

all_pred = np.concatenate(all_pred)
all_true = np.concatenate(all_true)

labels = ['HOMO', 'LUMO', 'Gap']
metrics = {}
for i, name in enumerate(labels):
    mae = mean_absolute_error(all_true[:, i], all_pred[:, i])
    r2 = r2_score(all_true[:, i], all_pred[:, i])
    print(f'{name}: MAE={mae:.4f} eV, R2={r2:.4f}')
    metrics[name] = {'mae': float(mae), 'r2': float(r2)}

metrics['best_params'] = bp
with open(f'{SAVE_DIR}/gps_2d_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved')

In [ ]:
# Save model weights + extract embeddings for fusion
print('Saving model weights...')
torch.save(model.state_dict(), f'{SAVE_DIR}/gps_2d_300k.pt')
print(f'  Saved gps_2d_300k.pt')

print('Extracting 2D embeddings...')
all_loader = DataLoader(graphs, batch_size=256, shuffle=False, num_workers=2)
embeddings_2d = []
with torch.no_grad():
    for batch_data in all_loader:
        batch_data = batch_data.to(device)
        emb = model.encode(batch_data.x, batch_data.edge_index,
                           batch_data.edge_attr, batch_data.batch)
        embeddings_2d.append(emb.cpu())
embeddings_2d = torch.cat(embeddings_2d, dim=0)
print(f'2D embeddings: {embeddings_2d.shape}')
torch.save(embeddings_2d, f'{SAVE_DIR}/gps_2d_embeddings.pt')

print('\nOutput files:')
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(f'{SAVE_DIR}/{f}') / 1e6
    print(f'  {f} ({size:.1f} MB)')
print('Done!')